# Corpus Overview

Distribution analysis of the 1,072-document annotation corpus. Token counts use `bert-base-uncased`. Noise status (`is_noise`) is shown throughout rather than filtered.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from transformers import AutoTokenizer

from nb_utils import setup_plots

plt = setup_plots()

NOISE_PALETTE = {True: '#d62728', False: '#1f77b4'}
NOISE_LABELS  = {True: 'Noise', False: 'Non-noise'}

In [ ]:
from nb_utils import load

FEATURES_CSV = '../features/features.csv'


In [ ]:
# corpus.parquet already has the numeric coercions and is_noise typing applied
# by the exporter. dolma_source/dolma_shard are the canonical names (see
# narradolma-catalog/SCHEMA.md); alias them back to the short forms this
# notebook uses.
df = load('corpus').rename(columns={'dolma_source': 'source',
                                    'dolma_shard':  'shard',
                                    'dolma_id':     'id'})

feat = pd.read_csv(FEATURES_CSV)
df = df.merge(feat, on='safe_instance_id', how='left')

unknown_mask = df['topic_classification'] == 'Unknown'
df.loc[unknown_mask, 'topic_classification'] = (
    'Shard: ' + df.loc[unknown_mask, 'source'].str.title()
)

print(f'Total rows:  {len(df)}')
print(f'  Noise:     {df.is_noise.sum()} ({df.is_noise.mean()*100:.1f}%)')
print(f'  Non-noise: {(~df.is_noise).sum()} ({(~df.is_noise).mean()*100:.1f}%)')
print(f'  Columns:   {list(df.columns)}')


In [ ]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
df['bert_tokens'] = df['sampled_text'].apply(
    lambda x: len(tokenizer.encode(str(x), add_special_tokens=False))
)
print('BERT token stats (sampled_text):')
print(df['bert_tokens'].describe().round(1))

## 1. Overview

In [ ]:
summary = {
    'Total documents':   len(df),
    'Noise (True)':      int(df['is_noise'].sum()),
    'Non-noise (False)': int((~df['is_noise']).sum()),
    'Unique folders':    df['folder'].nunique(),
    'Unique topics':     df['topic_classification'].nunique(),
    'Unique sources':    df['source'].nunique(),
    'Unique shards':     df['shard'].nunique(),
}
pd.DataFrame.from_dict(summary, orient='index', columns=['Count'])

## 2. Noise distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

noise_counts = df['is_noise'].value_counts().rename(NOISE_LABELS)
colors = [NOISE_PALETTE[k] for k in df['is_noise'].value_counts().index]
noise_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='white')
axes[0].set_title('is_noise counts')
axes[0].set_xlabel('')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                 str(int(bar.get_height())), ha='center', va='bottom', fontsize=10)

axes[1].pie(noise_counts, labels=noise_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90)
axes[1].set_title('is_noise proportion')

plt.suptitle('Noise status', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Folder distribution

In [ ]:
folder_noise = (df.groupby(['folder', 'is_noise'])
               .size().unstack(fill_value=0)
               .rename(columns=NOISE_LABELS))
folder_noise['Total'] = folder_noise.sum(axis=1)
folder_noise = folder_noise.sort_values('Total', ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
folder_noise[['Non-noise', 'Noise']].plot(
    kind='barh', stacked=True, ax=ax,
    color=[NOISE_PALETTE[False], NOISE_PALETTE[True]], edgecolor='white'
)
ax.set_title('Documents per folder')
ax.set_xlabel('Count')
ax.set_ylabel('')
for i, total in enumerate(folder_noise['Total']):
    ax.text(total + 1, i, str(total), va='center', fontsize=9)
plt.tight_layout()
plt.show()

print(folder_noise[['Non-noise', 'Noise', 'Total']].sort_values('Total', ascending=False))

## 4. Topic distribution

In [ ]:
topic_noise = (df.groupby(['topic_classification', 'is_noise'])
               .size().unstack(fill_value=0)
               .rename(columns=NOISE_LABELS))
topic_noise['Total'] = topic_noise.sum(axis=1)
topic_noise = topic_noise.sort_values('Total', ascending=True)

fig, ax = plt.subplots(figsize=(9, 9))
topic_noise[['Non-noise', 'Noise']].plot(
    kind='barh', stacked=True, ax=ax,
    color=[NOISE_PALETTE[False], NOISE_PALETTE[True]], edgecolor='white'
)
ax.set_title('Documents per topic')
ax.set_xlabel('Count')
ax.set_ylabel('')
for i, total in enumerate(topic_noise['Total']):
    ax.text(total + 1, i, str(total), va='center', fontsize=8)
plt.tight_layout()
plt.show()

print(topic_noise[['Non-noise', 'Noise', 'Total']].sort_values('Total', ascending=False))

In [ ]:
NAR_PALETTE = {'High (≥0.5)': '#2ca02c', 'Low (<0.5)': '#ff7f0e'}

nar_conf_group = df['narrative_confidence'].apply(
    lambda x: 'High (≥0.5)' if pd.notna(x) and x >= 0.5 else 'Low (<0.5)'
)
topic_nar = (df.groupby(['topic_classification', nar_conf_group])
               .size().unstack(fill_value=0))
for col in ['High (≥0.5)', 'Low (<0.5)']:
    if col not in topic_nar.columns:
        topic_nar[col] = 0
topic_nar['Total'] = topic_nar.sum(axis=1)
topic_nar = topic_nar.sort_values('Total', ascending=True)

fig, ax = plt.subplots(figsize=(9, 9))
topic_nar[['High (≥0.5)', 'Low (<0.5)']].plot(
    kind='barh', stacked=True, ax=ax,
    color=[NAR_PALETTE['High (≥0.5)'], NAR_PALETTE['Low (<0.5)']], edgecolor='white'
)
ax.set_title('Documents per topic by narrative confidence')
ax.set_xlabel('Count')
ax.set_ylabel('')
for i, total in enumerate(topic_nar['Total']):
    ax.text(total + 1, i, str(total), va='center', fontsize=8)
plt.tight_layout()
plt.show()

print(topic_nar[['High (≥0.5)', 'Low (<0.5)', 'Total']].sort_values('Total', ascending=False))

## 5. Narrative label & confidence

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# narrative label stacked bar
nar_noise = (df.groupby(['narrative_label', 'is_noise'])
             .size().unstack(fill_value=0)
             .rename(columns=NOISE_LABELS))
nar_noise[['Non-noise', 'Noise']].plot(
    kind='bar', ax=axes[0], stacked=True,
    color=[NOISE_PALETTE[False], NOISE_PALETTE[True]], edgecolor='white'
)
axes[0].set_title('Narrative label')
axes[0].tick_params(axis='x', rotation=15)
axes[0].set_ylabel('Count')

# narrative confidence KDE
for nv, lbl, clr in [(False, 'Non-noise', NOISE_PALETTE[False]),
                      (True,  'Noise',     NOISE_PALETTE[True])]:
    (df[df.is_noise == nv]['narrative_confidence']
       .dropna().plot(kind='kde', ax=axes[1], label=lbl, color=clr))
axes[1].set_title('Narrative confidence')
axes[1].set_xlabel('Confidence')
axes[1].legend()

# topic confidence KDE
for nv, lbl, clr in [(False, 'Non-noise', NOISE_PALETTE[False]),
                      (True,  'Noise',     NOISE_PALETTE[True])]:
    (df[df.is_noise == nv]['topic_confidence']
       .dropna().plot(kind='kde', ax=axes[2], label=lbl, color=clr))
axes[2].set_title('Topic confidence')
axes[2].set_xlabel('Confidence')
axes[2].legend()

plt.tight_layout()
plt.show()

## 6. Token length (BERT, sampled_text)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

for nv, lbl, clr in [(False, 'Non-noise', NOISE_PALETTE[False]),
                      (True,  'Noise',     NOISE_PALETTE[True])]:
    ax.hist(df[df.is_noise == nv]['bert_tokens'], bins=40,
            alpha=0.6, label=lbl, color=clr, edgecolor='white')

for q, ls, lbl in [(0.25, '--', 'Q1'), (0.50, '-', 'Median'), (0.75, '--', 'Q3')]:
    val = df['bert_tokens'].quantile(q)
    ax.axvline(val, color='black', linestyle=ls, linewidth=1.2,
               label=f'{lbl} = {val:.0f}')

ax.set_title('BERT token count distribution (bert-base-uncased)')
ax.set_xlabel('Token count')
ax.set_ylabel('Documents')
ax.legend()
plt.tight_layout()
plt.show()

print(df.groupby('is_noise')['bert_tokens']
      .describe().rename(index=NOISE_LABELS).round(1))

## 7. Automatic features

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

feature_configs = [
    ('sentiment_compound',    'Sentiment (compound)',     'kde'),
    ('concreteness_mean',     'Concreteness mean',        'kde'),
    ('past_tense_verb_rate',  'Past-tense verb rate',     'kde'),
    ('temporal_mention_rate', 'Temporal mention rate',    'hist'),
    ('pov_first_rate',        'POV first-person rate',    'kde'),
    ('pov_third_rate',        'POV third-person rate',    'kde'),
]

for ax, (col, title, ptype) in zip(axes.flat, feature_configs):
    for nv, lbl, clr in [(False, 'Non-noise', NOISE_PALETTE[False]),
                          (True,  'Noise',     NOISE_PALETTE[True])]:
        vals = df[df.is_noise == nv][col].dropna().astype(float)
        if ptype == 'kde':
            vals.plot(kind='kde', ax=ax, label=lbl, color=clr)
        else:
            ax.hist(vals, bins=25, alpha=0.6, label=lbl, color=clr, edgecolor='white')
    ax.set_title(title)
    ax.legend(fontsize=8)

plt.suptitle('Automatic features by noise status', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Dominant POV bar chart
fig, ax = plt.subplots(figsize=(7, 4))
pov_noise = (df.groupby(['pov_dominant', 'is_noise'])
             .size().unstack(fill_value=0)
             .rename(columns=NOISE_LABELS))
pov_noise[['Non-noise', 'Noise']].plot(
    kind='bar', ax=ax, stacked=True,
    color=[NOISE_PALETTE[False], NOISE_PALETTE[True]], edgecolor='white'
)
ax.set_title('Dominant point-of-view by noise status')
ax.set_xlabel('POV')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

## 8. Event and verb counts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, col, title in zip(axes,
                           ['event_count', 'verb_count'],
                           ['Event count (per passage)', 'Verb count (per passage)']):
    max_val = int(df[col].dropna().astype(float).max())
    bins = list(range(0, max_val + 2))
    for nv, lbl, clr in [(False, 'Non-noise', NOISE_PALETTE[False]),
                          (True,  'Noise',     NOISE_PALETTE[True])]:
        vals = df[df.is_noise == nv][col].dropna().astype(float)
        ax.hist(vals, bins=bins, alpha=0.6, label=lbl, color=clr, edgecolor='white')
    ax.set_title(title)
    ax.set_xlabel('Count')
    ax.set_ylabel('Documents')
    ax.legend()

plt.tight_layout()
plt.show()

print(df.groupby('is_noise')[['event_count', 'verb_count']]
      .describe().rename(index=NOISE_LABELS).round(1))

## 9. Correlation heatmap (numeric features)

In [ ]:
num_cols = [
    'bert_tokens', 'narrative_confidence', 'topic_confidence',
    'event_count', 'verb_count',
    'sentiment_compound', 'sentiment_pos', 'sentiment_neg',
    'concreteness_mean', 'concreteness_coverage',
    'past_tense_verb_rate', 'temporal_mention_rate', 'temporal_mention_count',
    'pov_first_rate', 'pov_second_rate', 'pov_third_rate',
]
corr = df[num_cols].astype(float).corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.4, ax=ax, annot_kws={'size': 7})
ax.set_title('Feature correlation matrix')
plt.tight_layout()
plt.show()